In [ ]:
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from matplotlib import pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import itertools
import tensorflow as tf


# Example: Q_MAX river flow
q_min = pd.read_csv("q_min.csv")

# Optional: clean column names
q_min.columns = q_min.columns.str.strip().str.lower()


#def smooth_loss(y_true, y_pred):
#    mse = tf.reduce_mean(tf.square(y_true - y_pred))
#    smooth = tf.reduce_mean(tf.square(y_pred[1:] - y_pred[:-1]))
#    return mse + 0.05 * smooth

def smooth_loss(y_true, y_pred):
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    penalty = tf.reduce_mean(tf.square(y_pred))
    return mse + 1e-6 * penalty

def prepare_time_series(df, value_name='flow'):
    months = ["jan","feb","mar","apr","maj","jun","jul","avg","sep","okt","nov","dec"]
    existing_months = [m for m in months if m in df.columns]

    # Convert to numeric and fill missing
    df[existing_months] = df[existing_months].apply(pd.to_numeric, errors='coerce')
    df[existing_months] = df[existing_months].fillna(method='ffill').fillna(method='bfill')

    # Melt to long format
    ts = df.melt(id_vars='year', value_vars=existing_months, var_name='month', value_name=value_name)
    ts['date'] = pd.to_datetime(ts['year'].astype(str) + '-' + ts['month'], format='%Y-%b', errors='coerce')
    ts = ts.dropna(subset=['date'])
    ts = ts.set_index('date').sort_index()
    ts = ts.dropna(subset=[value_name])
    
    # Ensure non-negative
    ts[value_name] = ts[value_name].clip(lower=0)
    
    return ts


def prepare_ml_data(ts, window_size=24, split_date='2014-01-01'):
    
    ts = ts.copy()
    ts['flow'] = ts['flow'].clip(lower=0)

    # Seasonal encoding
    ts['sin_month'] = np.sin(2 * np.pi * ts.index.month / 12)
    ts['cos_month'] = np.cos(2 * np.pi * ts.index.month / 12)

    flows = ts['flow'].values
    sin_m = ts['sin_month'].values
    cos_m = ts['cos_month'].values
    dates = ts.index

    X, y, idx = [], [], []
    for i in range(window_size, len(flows)):
        lag_vals = flows[i-window_size:i]
        features = np.concatenate([lag_vals, [sin_m[i], cos_m[i]]])
        
        X.append(features)
        y.append(flows[i])
        idx.append(dates[i])

    X = np.array(X)
    y = np.array(y)
    idx = pd.to_datetime(idx)

    # Log transform
    y_log = np.log1p(y)

    # Train/test split
    split_date = pd.to_datetime(split_date)
    split_idx = np.where(idx < split_date)[0]
    split = split_idx[-1] + 1 if len(split_idx) > 0 else 0

    return {
        "X_train": X[:split],
        "y_train": y_log[:split],
        "X_test": X[split:],
        "y_test": y[split:],        # original scale
        "idx_test": idx[split:]
    }

def nse(y_true, y_pred):
    """Nash-Sutcliffe Efficiency"""
    denom = np.sum((y_true - np.mean(y_true))**2)
    if denom == 0:
        return 0
    return 1 - np.sum((y_true - y_pred)**2) / denom

def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    nse = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    return rmse, mae, nse

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GRU, Dense, MaxPooling1D, Dropout

from tensorflow.keras import regularizers

def build_cnn_gru(input_shape):
    
    model = Sequential([
        Conv1D(32, 3, activation='relu',
               kernel_regularizer=regularizers.l2(1e-4),
               input_shape=input_shape),
        MaxPooling1D(2),
        Dropout(0.3),

        GRU(32, return_sequences=False,
            kernel_regularizer=regularizers.l2(1e-4)),
        Dropout(0.3),

        Dense(1)
    ])

    model.compile(optimizer='adam', loss=smooth_loss) #'mse')
    return model

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

def train_cnn_gru(X_train, y_train):

    tscv = TimeSeriesSplit(n_splits=5)
    scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)

        X_tr = X_tr.reshape((X_tr.shape[0], X_tr.shape[1], 1))
        X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))

        model = build_cnn_gru((X_tr.shape[1], 1))

        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True
        )

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=500,
            batch_size=32,
            verbose=0,
            callbacks=[early_stop]
        )

        y_pred = np.expm1(model.predict(X_val).flatten())
        y_true = np.expm1(y_val)

        scores.append(nse(y_true, y_pred))

    print("CNN-GRU CV NSE (mean):", np.mean(scores))

    # ✅ FINAL TRAIN ON FULL DATA
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    X_scaled = X_scaled.reshape((X_scaled.shape[0], X_scaled.shape[1], 1))

    final_model = build_cnn_gru((X_scaled.shape[1], 1))

    final_model.fit(
        X_scaled, y_train,
        epochs=500,
        batch_size=32,
        verbose=0
    )

    return final_model, scaler



import matplotlib.pyplot as plt
import numpy as np

def test_cnn_gru(model, scaler, data):

    X_test  = data["X_test"]
    y_test  = data["y_test"]
    idx     = data["idx_test"]

    # -----------------------------
    # 1. Scale + reshape
    # -----------------------------
    X_test = scaler.transform(X_test)
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    # -----------------------------
    # 2. Predict
    # -----------------------------
    y_pred = np.expm1(model.predict(X_test).flatten())
    y_pred = np.maximum(y_pred, 0)

    # -----------------------------
    # 3. Align
    # -----------------------------
    min_len = min(len(y_test), len(y_pred))
    y_test = y_test[:min_len]
    y_pred = y_pred[:min_len]
    idx = idx[:min_len]

    # -----------------------------
    # 4. Metrics
    # -----------------------------
    rmse, mae, nse_val = compute_metrics(y_test, y_pred)

    print("\n==============================")
    print("CNN-GRU Performance")
    print("==============================")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE : {mae:.2f}")
    print(f"NSE : {nse_val:.4f}")

    # -----------------------------
    # 5. Plot 1: Time series
    # -----------------------------
    plt.figure(figsize=(14,5))
    plt.plot(idx, y_test, label='Actual')
    plt.plot(idx, y_pred, label='CNN-GRU', linestyle='--', marker='x')
    plt.title("CNN-GRU Prediction (2014–2025)")
    plt.xlabel("Date")
    plt.ylabel("Flow")
    plt.legend()
    plt.grid(True)
    plt.show()

    # -----------------------------
    # 6. Plot 2: Residuals over time
    # -----------------------------
    residuals = y_test - y_pred

    plt.figure(figsize=(14,4))
    plt.plot(idx, residuals)
    plt.axhline(0)
    plt.title("Residuals (Actual - Predicted)")
    plt.xlabel("Date")
    plt.ylabel("Error")
    plt.grid(True)
    plt.show()

    # -----------------------------
    # 7. Plot 3: Error distribution
    # -----------------------------
    plt.figure(figsize=(6,4))
    plt.hist(residuals, bins=30)
    plt.title("Residual Distribution")
    plt.xlabel("Error")
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.show()

    return y_pred

In [ ]:
# Example: Q_MAX river flow
q_min = pd.read_csv("q_min.csv")

# Optional: clean column names
q_min.columns = q_min.columns.str.strip().str.lower()

ts_qmin = prepare_time_series(q_min)
data = prepare_ml_data(ts_qmin)

## train-test split 
X_train = data["X_train"]
y_train = data["y_train"]

X_test  = data["X_test"]
y_test  = data["y_test"]
idx     = data["idx_test"]

cnn_gru_model, cnn_gru_scaler = train_cnn_gru(X_train, y_train)
y_pred_cnn_gru = test_cnn_gru(cnn_gru_model, cnn_gru_scaler, data)